# pyrosm smoke test

Checks, before wiring `pyrosm` into `src/prepare_data.py`, that its output columns for buildings/landuse/roads/parking are close enough to what the notebook's existing feature-engineering code expects (which was written against `osmnx.features_from_place`'s column names: `building`, `landuse`, `highway`, `amenity`, `addr:street`, `building:levels`, `shop`, `website`, `brand`, `name`, `source`).

Run this with the `berlin-buildings` conda env active. Small bbox over central Mitte so it's fast.

In [1]:
from pyrosm import OSM

# small bbox over central Mitte, fast to parse
bbox = [13.38, 52.51, 13.40, 52.53]
osm = OSM("../data/raw/berlin-260805.osm.pbf", bounding_box=bbox)

In [2]:
buildings = osm.get_buildings()
print(buildings.shape)
print(sorted(buildings.columns.tolist()))

EXPECTED_BUILDING_COLS = {"building", "addr:street", "building:levels", "shop", "website", "amenity", "name", "source"}
missing = EXPECTED_BUILDING_COLS - set(buildings.columns)
print("missing expected columns:", missing if missing else "none")

/home/labdoo/anaconda3/envs/berlin-buildings/lib/python3.11/site-packages/pyrosm/pyrosm.py:565: UserWarning: 9 relation(s) extend beyond the bounding box and were returned with incomplete geometry (some member ways fall outside the box). Pass complete_relations=True to OSM(...) to fetch their full member set.
  gdf = get_building_data(


(1926, 36)
['addr:city', 'addr:country', 'addr:housename', 'addr:housenumber', 'addr:place', 'addr:postcode', 'addr:street', 'amenity', 'building', 'building:levels', 'building:material', 'building:min_level', 'building:use', 'changeset', 'email', 'geometry', 'height', 'id', 'internet_access', 'name', 'office', 'opening_hours', 'operator', 'osm_type', 'phone', 'ref', 'shop', 'source', 'start_date', 'tags', 'timestamp', 'url', 'version', 'visible', 'website', 'wikipedia']
missing expected columns: none


In [3]:
# 'brand' isn't a first-class column above -- check whether it survives inside pyrosm's
# leftover 'tags' dict column (untranslated tags), which would mean has_brand can still
# be computed, just via buildings['tags'].apply(lambda t: 'brand' in t) instead of
# buildings['brand'].notna()
has_brand_in_tags = buildings["tags"].dropna().apply(lambda t: isinstance(t, dict) and "brand" in t).sum()
print("rows with brand inside tags dict:", has_brand_in_tags)
print(buildings["tags"].dropna().iloc[0] if buildings["tags"].notna().any() else "no non-null tags examples")

rows with brand inside tags dict: 0
{"addr:suburb":"Mitte","architect":"Johann Arnold Nering;Martin Gr\u00FCnberg;Andreas Schl\u00FCter;Jean de Bodt","architect:wikidata":"Q286441;Q102055;Q500465;Q102284","building:architecture":"baroque","building:colour":"#f1dcd0","construction_date":"1695..1706","heritage":"4","heritage:operator":"lda","heritage:website":"https://denkmaldatenbank.berlin.de/daobj.php?obj_dok_nr=09095948","image":"https://upload.wikimedia.org/wikipedia/commons/f/f4/Zeughaus_Berlin_2012.jpg","lda:criteria":"Baudenkmal","name:da":"T\u00F8jhuset","name:de":"Zeughaus","name:en":"Arsenal","name:fr":"Arsenal","name:it":"Arsenale","name:nb":"T\u00F8yhuset","ref:lda":"09095948,T","roof:colour":"#7a8682","wheelchair":"limited","wikidata":"Q196970","wikimedia_commons":"Category:Zeughaus (Berlin)"}


In [4]:
landuse = osm.get_landuse()
print(None if landuse is None else landuse.shape)
print(None if landuse is None else sorted(landuse.columns.tolist()))

(400, 9)
['changeset', 'construction', 'geometry', 'id', 'landuse', 'osm_type', 'tags', 'timestamp', 'version']


/home/labdoo/anaconda3/envs/berlin-buildings/lib/python3.11/site-packages/pyrosm/pyrosm.py:655: UserWarning: 2 relation(s) extend beyond the bounding box and were returned with incomplete geometry (some member ways fall outside the box). Pass complete_relations=True to OSM(...) to fetch their full member set.
  gdf = get_landuse_data(


In [5]:
network = osm.get_network(network_type="all")
print(network.shape)
print(sorted(network.columns.tolist()))
print()
print("highway value_counts:")
print(network["highway"].value_counts())
print()
print("geom types:", network.geom_type.value_counts().to_dict())

# road_bucket() only cares about these -- confirm they're all present as line geometries
NEEDED_HIGHWAY_VALUES = {"residential", "living_street", "primary", "secondary", "tertiary", "motorway", "trunk", "service"}
present = set(network["highway"].dropna().unique())
print("needed highway values present:", NEEDED_HIGHWAY_VALUES & present)
print("needed highway values MISSING:", NEEDED_HIGHWAY_VALUES - present)

(4476, 33)
['access', 'bicycle', 'bicycle_road', 'bridge', 'cycleway', 'est_width', 'foot', 'footway', 'geometry', 'highway', 'id', 'lanes', 'length', 'lit', 'maxspeed', 'motor_vehicle', 'motorcar', 'name', 'oneway', 'oneway:bicycle', 'osm_type', 'psv', 'ref', 'segregated', 'service', 'sidewalk', 'smoothness', 'surface', 'tags', 'timestamp', 'tunnel', 'version', 'width']

highway value_counts:
highway
footway           2457
service            726
residential        541
steps              217
tertiary           198
secondary          145
primary             97
pedestrian          23
path                23
corridor            23
living_street       16
cycleway             5
elevator             2
secondary_link       1
primary_link         1
bus_stop             1
Name: count, dtype: int64

geom types: {'MultiLineString': 4476}
needed highway values present: {'tertiary', 'living_street', 'service', 'secondary', 'primary', 'residential'}
needed highway values MISSING: {'motorway', 'trunk'

In [6]:
parking = osm.get_data_by_custom_criteria(
    custom_filter={"amenity": ["parking", "parking_space"]},
    filter_type="keep",
    keep_nodes=False, keep_ways=True, keep_relations=True,
)
print(None if parking is None else parking.shape)
print(None if parking is None else sorted(parking.columns.tolist()))
print(None if parking is None else parking.geom_type.value_counts().to_dict())

(254, 19)
['addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'amenity', 'geometry', 'id', 'name', 'opening_hours', 'operator', 'osm_type', 'parking', 'tags', 'taxi', 'timestamp', 'version', 'visible', 'website']
{'Polygon': 254}
